In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent  
sys.path.append(str(repo_root))

In [52]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, learning_curve, LearningCurveDisplay
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
from src.data.Segment_Slicer import SegmentSlicer 
SegmentSlicer = SegmentSlicer()

In [4]:
df= pd.read_parquet(repo_root / 'data' / 'processed' / 'reunion_segments_cleaned.parquet')

In [5]:
print(f"📊 Dataset: {len(df)} segments")
print(f"   Running: {(df['activity_type']=='Run').sum()}")
print(f"   Cycling: {(df['activity_type']=='Ride').sum()}")

📊 Dataset: 4768 segments
   Running: 2502
   Cycling: 2266


In [6]:
Train, Test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['activity_type'])
Train, Val = train_test_split(Train, test_size=0.25, random_state=42, stratify=Train['activity_type'])
print(f"Train: {len(Train)}, Val: {len(Val)}, Test: {len(Test)}")

Train: 2860, Val: 954, Test: 954


In [7]:
# Séparer Ride et Run
Train_ride = Train[Train['activity_type'] == 'Ride'].copy()
Train_run = Train[Train['activity_type'] == 'Run'].copy()

Val_ride = Val[Val['activity_type'] == 'Ride'].copy()
Val_run = Val[Val['activity_type'] == 'Run'].copy()

Test_ride = Test[Test['activity_type'] == 'Ride'].copy()
Test_run = Test[Test['activity_type'] == 'Run'].copy()

In [31]:
Train_Val_Test_ride = pd.concat([Train_ride, Val_ride, Test_ride], ignore_index=True)
Train_Val_Test_run = pd.concat([Train_run, Val_run, Test_run], ignore_index=True)

In [8]:
T1_active_learning_ride = pd.read_csv(repo_root / 'data' / 'processed' / 'T1_active_learning_threshold_7_ride.csv')
T1_road_naming_ride = pd.read_csv(repo_root / 'data' / 'processed' / 'T1_road_naming_ride.csv')
segments_manually_labeled = pd.read_csv(repo_root / 'data' / 'processed' / 'segments_manually_labeled.csv')
T1_segments_manually_labeled_ride = segments_manually_labeled[(segments_manually_labeled['technicality'] == 1) |(segments_manually_labeled['segment_id'].isin(df[df['activity_type'] == 'Ride']['segment_id']))].copy()

T1_ride = pd.concat([T1_active_learning_ride, T1_road_naming_ride, T1_segments_manually_labeled_ride]).drop_duplicates().reset_index(drop=True)
print(f"Total T1 Ride segments labeled: {len(T1_ride)}")

Total T1 Ride segments labeled: 547


In [36]:
X_train_ride_T1 = Train_ride.merge(T1_ride[['segment_id', 'technicality']], on='segment_id', how='inner')
y_train_ride_T1 = X_train_ride_T1['best_time'].fillna(0).astype(int)

X_val_ride_T1 = Val_ride.merge(T1_ride[['segment_id', 'technicality']], on='segment_id', how='inner')
y_val_ride_T1 = X_val_ride_T1['best_time'].fillna(0).astype(int)

### Baseline Model

In [38]:
X_train_ride_T1_bm = X_train_ride_T1.loc[:, ['distance', 'elevation_gain', 'elevation_low', 'elevation_high']].copy()
X_val_ride_T1_bm = X_val_ride_T1.loc[:, ['distance', 'elevation_gain', 'elevation_low', 'elevation_high']].copy()
y_train_ride_T1_bm = y_train_ride_T1.copy()
y_val_ride_T1_bm = y_val_ride_T1.copy()


In [39]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, min_samples_leaf=10))
])

In [40]:
pipeline.fit(X_train_ride_T1_bm.select_dtypes(include=[np.number]), y_train_ride_T1_bm)
y_val_pred = pipeline.predict(X_val_ride_T1_bm.select_dtypes(include=[np.number]))
print(f"Validation RMSE: {root_mean_squared_error(y_val_ride_T1_bm, y_val_pred):.2f}")
print(f"Validation MAE: {mean_absolute_error(y_val_ride_T1_bm, y_val_pred):.2f} s")

train_size_abs, train_scores, test_scores = learning_curve(
    pipeline, X_train_ride_T1_bm.select_dtypes(include=[np.number]), y_train_ride_T1_bm,
    cv=5, scoring='neg_root_mean_squared_error', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)

# Create subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=("Learning Curve", "Prediction vs Actual"))

# Plot learning curve
train_sizes = np.linspace(0.1, 1.0, 10) * len(X_train_ride_T1_bm)
train_scores_mean = -np.mean(train_scores, axis=1)
test_scores_mean = -np.mean(test_scores, axis=1)

fig.add_trace(
    go.Scatter(
        x=train_sizes,
        y=train_scores_mean,
        mode='lines+markers',
        name='Training Score',
        line=dict(color='blue')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=train_sizes,
        y=test_scores_mean,
        mode='lines+markers',
        name='Validation Score',
        line=dict(color='red')
    ),
    row=1, col=1
)

fig.update_xaxes(title_text="Training Examples", row=1, col=1)
fig.update_yaxes(title_text="MAE", row=1, col=1)

# Merge segment_id back for visualization purposes
X_val_with_segment_id = Val_ride.merge(T1_ride[['segment_id', 'technicality']], on='segment_id', how='inner')
X_val_with_segment_id = X_val_with_segment_id.loc[:, ['segment_id', 'best_time']].drop_duplicates()

# Plot prediction vs actual values
fig.add_trace(
    go.Scatter(
        x=y_val_ride_T1_bm,
        y=y_val_pred,
        mode='markers',
        name='Predictions',
        text=X_val_with_segment_id['segment_id'],  # Use segment_id for hover text
        customdata=X_val_with_segment_id['segment_id'],
        hovertemplate="<b>Segment ID:</b> %{text}<br>" +
                      "<b>Actual:</b> %{x:.2f}<br>" +
                      "<b>Predicted:</b> %{y:.2f}<extra></extra>"
    ),
    row=1, col=2
)

# Add a line for perfect predictions
fig.add_trace(
    go.Scatter(
        x=[y_val_ride_T1_bm.min(), y_val_ride_T1_bm.max()],
        y=[y_val_ride_T1_bm.min(), y_val_ride_T1_bm.max()],
        mode='lines',
        line=dict(dash='dash', color='black'),
        showlegend=False
    ),
    row=1, col=2
)

fig.update_xaxes(title_text="Actual", row=1, col=2)
fig.update_yaxes(title_text="Predicted", row=1, col=2)

# Update layout
fig.update_layout(
    height=500,
    width=1000,
    title_text=f"Model Evaluation",
    hovermode='closest'
)

fig.show()

Validation RMSE: 288.88
Validation MAE: 89.10 s


### Model 1

In [42]:
def extract_features_from_sections(sections, segment_id=None):
    """
    Extrait des features simples à partir des sections d'un segment.
    
    Input: sections (list of dict) - output de segment_slicer.cut_segment()
    Output: dict of features
    """
    
    if not sections or len(sections) == 0:
        return None
    
    # ========== Features Basiques ==========
    total_distance = sum(s['distance'] for s in sections) / 1000  # en km
    total_elevation_gain = sum(s['elevation_gain'] for s in sections)
    total_elevation_loss = sum(s['elevation_loss'] for s in sections)
    
    # Grades
    all_grades = [s['grade'] for s in sections]
    avg_grade = np.mean(all_grades)
    max_grade = max(s['max_grade'] for s in sections)
    min_grade = min(s['min_grade'] for s in sections)
    
    # ========== Features d'Ordre (Capture la Séquence) ==========
    
    # 1. Distribution des montées (early vs late)
    early_third_distance = total_distance * 1000 * 0.33
    late_third_distance = total_distance * 1000 * 0.67
    
    early_climb_gain = sum(s['elevation_gain'] for s in sections 
                           if s['start_distance'] < early_third_distance)
    late_climb_gain = sum(s['elevation_gain'] for s in sections 
                          if s['start_distance'] > late_third_distance)
    
    early_climb_ratio = early_climb_gain / (total_elevation_gain + 1e-6)
    late_climb_ratio = late_climb_gain / (total_elevation_gain + 1e-6)
    
    # 2. Grade pondéré par position (effet fatigue)
    weighted_grade = 0
    for i, s in enumerate(sections):
        position_weight = 1 + (i / len(sections)) * 0.5  # 1.0 à 1.5x
        weighted_grade += s['grade'] * position_weight * s['distance']
    weighted_grade /= (total_distance * 1000)
    
    # 3. Position de la section la plus dure
    hardest_idx = np.argmax([s['grade'] * s['distance'] for s in sections])
    hardest_section_position = hardest_idx / len(sections)  # 0 à 1
    
    # 4. Variabilité du terrain
    grade_variance = np.mean([s['grade_variance'] for s in sections])
    
    # 5. Stats par tiers du segment
    first_third = [s for s in sections if s['start_distance'] < early_third_distance]
    middle_third = [s for s in sections 
                    if early_third_distance <= s['start_distance'] <= late_third_distance]
    last_third = [s for s in sections if s['start_distance'] > late_third_distance]
    
    first_third_avg_grade = np.mean([s['grade'] for s in first_third]) if first_third else 0
    middle_third_avg_grade = np.mean([s['grade'] for s in middle_third]) if middle_third else 0
    last_third_avg_grade = np.mean([s['grade'] for s in last_third]) if last_third else 0
    
    # 6. Compter les types de sections
    n_climbs = sum(1 for s in sections if s['type'] in ['climb', 'uphill'])
    n_descents = sum(1 for s in sections if s['type'] in ['descent', 'downhill'])
    n_flats = sum(1 for s in sections if s['type'] == 'flat')

    # 7. Features lié au data leakage
    best_time = df.loc[df['segment_id'] == segment_id, 'best_time'].iloc[0]
    avg_top_10_time = df.loc[df['segment_id'] == segment_id, 'average_top_10_time'].iloc[0]
    total_effort_count = df.loc[df['segment_id'] == segment_id, 'total_effort_count'].iloc[0]
    inv_total_effort_count = 1 / (total_effort_count + 1e-6)    
    
    # ========== Assembler le dictionnaire de features ==========
    features = {
        # Basiques
        'total_distance_km': total_distance,
        'total_elevation_gain': total_elevation_gain,
        'total_elevation_loss': total_elevation_loss,
        'avg_grade': avg_grade,
        'max_grade': max_grade,
        'min_grade': min_grade,
        'grade_variance': grade_variance,
        
        # Ordre et fatigue
        'early_climb_ratio': early_climb_ratio,
        'late_climb_ratio': late_climb_ratio,
        'weighted_grade': weighted_grade,
        'hardest_section_position': hardest_section_position,
        
        # Tiers
        'first_third_avg_grade': first_third_avg_grade,
        'middle_third_avg_grade': middle_third_avg_grade,
        'last_third_avg_grade': last_third_avg_grade,
        
        # Comptages
        'n_sections': len(sections),
        'n_climbs': n_climbs,
        'n_descents': n_descents,
        'n_flats': n_flats,

        # Data leakage
        #'best_time': best_time,
        #'avg_top_10_time': avg_top_10_time,
        #'total_effort_count': total_effort_count,
        #'inv_total_effort_count': inv_total_effort_count
    }
    
    return features

In [43]:
def extract_features_for_dataframe(df, sections_dict):
    """
    Extrait les features pour tous les segments d'une DataFrame.
    
    Input:
        - df: DataFrame avec les segments (doit avoir une colonne 'id')
        - sections_dict: dict {segment_id: sections} où sections = output de cut_segment
    
    Output:
        - features_df: DataFrame avec les features
        - valid_indices: indices des segments traités avec succès
    """
    
    features_list = []
    valid_indices = []
    
    for idx, row in df.iterrows():
        segment_id = row['segment_id']
        
        # Vérifier si on a les sections pour ce segment
        if segment_id not in sections_dict:
            print(f"Warning: No sections found for segment {segment_id}")
            continue
        
        sections = sections_dict[segment_id]
        
        # Extraire les features
        features = extract_features_from_sections(sections, segment_id=segment_id)
        
        if features is not None:
            features_list.append(features)
            valid_indices.append(idx)
        else:
            print(f"Warning: Could not extract features for segment {segment_id}")
    
    # Créer la DataFrame de features
    features_df = pd.DataFrame(features_list, index=valid_indices)
    
    print(f"✓ Extracted features for {len(features_df)} / {len(df)} segments")
    
    return features_df, valid_indices

In [35]:
sections_dict = {}
for idx, row in Train_Val_Test_ride.iterrows():
    segment_id = row['segment_id']
    # Charger altitude_profile, distance_profile, coordinates
    sections = SegmentSlicer.cut_segment(row['altitude_profile'], row['distance_profile'], row['coordinates'])
    sections_dict[segment_id] = sections

In [49]:
X_train_ride_2, valid_idx_train_2 = extract_features_for_dataframe(X_train_ride_T1, sections_dict)
y_train_ride_2 = y_train_ride_T1.loc[valid_idx_train_2]

X_val_ride_2, valid_idx_val_2 = extract_features_for_dataframe(X_val_ride_T1, sections_dict)
y_val_ride_2 = y_val_ride_T1.copy()

✓ Extracted features for 316 / 316 segments
✓ Extracted features for 128 / 128 segments


In [65]:
pipeline2 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(
        alpha=50
    ))
])

In [66]:
pipeline2.fit(X_train_ride_2.select_dtypes(include=[np.number]), y_train_ride_2)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,50
,fit_intercept,True
,copy_X,True
,max_iter,None


In [69]:
def plotting_pred_vs_actual_LC(pipeline, X_train, y_train, X_val, y_val, T1_df, Val_df):
    
    pipeline.fit(X_train.select_dtypes(include=[np.number]), y_train)
    y_val_pred = pipeline.predict(X_val.select_dtypes(include=[np.number]))
    print(f"Validation RMSE: {root_mean_squared_error(y_val, y_val_pred):.2f}")
    print(f"Validation MAE: {mean_absolute_error(y_val, y_val_pred):.2f} s")

    train_size_abs, train_scores, test_scores = learning_curve(
        pipeline, X_train.select_dtypes(include=[np.number]), y_train,
        cv=5, scoring='neg_root_mean_squared_error', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )

    fig = make_subplots(rows=1, cols=2, subplot_titles=("Learning Curve", "Prediction vs Actual"))

    # Plot learning curve
    train_sizes = np.linspace(0.1, 1.0, 10) * len(X_train_ride_2)
    train_scores_mean = -np.mean(train_scores, axis=1)
    test_scores_mean = -np.mean(test_scores, axis=1)

    fig.add_trace(
        go.Scatter(
            x=train_sizes,
            y=train_scores_mean,
            mode='lines+markers',
            name='Train Score',
            line=dict(color='blue')
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=train_sizes,
            y=test_scores_mean,
            mode='lines+markers',
            name='Val Score',
            line=dict(color='red')
        ),
        row=1, col=1
    )

    fig.update_xaxes(title_text="Training Examples", row=1, col=1)
    fig.update_yaxes(title_text="MAE", row=1, col=1)

    # Merge segment_id back for visualization purposes
    X_val_with_segment_id = Val_df.merge(T1_df[['segment_id', 'technicality']], on='segment_id', how='inner')
    X_val_with_segment_id = X_val_with_segment_id.loc[:, ['segment_id', 'best_time']].drop_duplicates()

    # Plot prediction vs actual values
    fig.add_trace(
        go.Scatter(
            x=y_val,
            y=y_val_pred,
            mode='markers',
            name='Predictions',
            text=X_val_with_segment_id['segment_id'],  # Use segment_id for hover text
            customdata=X_val_with_segment_id['segment_id'],
            hovertemplate="<b>Segment ID:</b> %{text}<br>" +
                        "<b>Actual:</b> %{x:.2f}<br>" +
                        "<b>Predicted:</b> %{y:.2f}<extra></extra>"
        ),
        row=1, col=2
    )

    # Add a line for perfect predictions
    fig.add_trace(
        go.Scatter(
            x=[y_val.min(), y_val.max()],
            y=[y_val.min(), y_val.max()],
            mode='lines',
            line=dict(dash='dash', color='black'),
            showlegend=False
        ),
        row=1, col=2
    )

    fig.update_xaxes(title_text="Actual", row=1, col=2)
    fig.update_yaxes(title_text="Predicted", row=1, col=2)

    # Update layout
    fig.update_layout(
        height=500,
        width=1000,
        title_text=f"Model Evaluation",
        hovermode='closest'
    )

    fig.show()
    return None

In [70]:
plotting_pred_vs_actual_LC(pipeline2, X_train_ride_2, y_train_ride_2, X_val_ride_2, y_val_ride_2, T1_ride, Val_ride)

Validation RMSE: 116.91
Validation MAE: 62.88 s
